# Executed MINI assembly qualification — 2026-10-23

Executed in Edge Colab from the pinned public notebook at 7a3ae5c11331c1238f023b0e51e11e147158f2fb. Main-cell output is reconstructed from the complete retained stage logs and receipt. The numerical run and raw-state export completed successfully. Transient archive-transfer and later display-only cells are omitted. A subsequent Colab HTML/JavaScript output-rendering error and browser-control stalls are documented separately; they did not change the completed numerical results or archives.


In [1]:
from pathlib import Path
import datetime, hashlib, json, os, platform, subprocess, sys, traceback, zipfile

CODE_COMMIT = "584785c32caadd3c6bc42fef712b3762789add85"
REFERENCE_COMMIT = "b2dacf79a8d0cdb93fbd34e5d78b19d4ee06d534"
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
ROOT = Path("/content") / ("mini_assembly_" + STAMP)
ROOT.mkdir()
REPO = ROOT / "CardiMech"
EVIDENCE = ROOT / "evidence"
EVIDENCE.mkdir()
STAGES = []
ENV = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}
def stage(name, command, cwd=None, required=True):
    started = datetime.datetime.now(datetime.timezone.utc)
    result = subprocess.run(command, cwd=cwd, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    (EVIDENCE / (name + ".log")).write_text(result.stdout)
    STAGES.append({"name": name, "command": command, "returncode": result.returncode,
                   "started_utc": started.isoformat(),
                   "elapsed_seconds": (datetime.datetime.now(datetime.timezone.utc)-started).total_seconds()})
    (EVIDENCE / "stages.json").write_text(json.dumps(STAGES, indent=2))
    print(name, "returncode", result.returncode, flush=True)
    print(result.stdout[-4000:], flush=True)
    if required:
        result.check_returncode()
    return result

failure = None
try:
    # Colab's distribution can lack ensurepip; use the PyPI virtualenv seeder.
    # Avoid naming any diagnostic file profile.py, which shadows the stdlib module.
    stage("virtualenv-install", [sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
    stage("venv", [sys.executable, "-m", "virtualenv", str(ROOT / "venv")])
    PY = str(ROOT / "venv/bin/python")
    stage("clone", ["git", "clone", "https://github.com/Virelion-Biotech/Virelion-CardiMech.git", str(REPO)])
    stage("checkout", ["git", "checkout", CODE_COMMIT], cwd=REPO)
    stage("install", [PY, "-m", "pip", "install", "-q", "-e", str(REPO) + "[dev,validation,spatial]"])
    INFER = ROOT / "CardiInfer"
    stage("infer-clone", ["git", "clone", "https://github.com/Virelion-Biotech/Virelion-CardiInfer.git", str(INFER)])
    stage("infer-checkout", ["git", "checkout", "15a7bd5bf4c23784245b361c017a376572766254"], cwd=INFER)
    stage("infer-install", [PY, "-m", "pip", "install", "-q", "-e", str(INFER)])
    stage("dependencies", [PY, "-m", "pip", "freeze"])
    stage("lint", [PY, "-m", "ruff", "check", "src", "tests", "scripts"], cwd=REPO, required=False)
    stage("format", [PY, "-m", "ruff", "format", "--check", "src", "tests", "scripts"], cwd=REPO, required=False)
    stage("suite", [PY, "-m", "pytest", "-q", "--junitxml=" + str(EVIDENCE / "junit.xml"),
                    "--cov=cardimech", "--cov-fail-under=80"], cwd=REPO, required=False)
    stage("comparison", [PY, "scripts/compare_mini_assembly.py", "--output",
                         str(EVIDENCE / "comparison")], cwd=REPO, required=False)
    stage("qualification", [PY, "scripts/verify_mixed_locking.py", "--output",
                            str(EVIDENCE / "qualification")], cwd=REPO, required=False)
except Exception:
    failure = traceback.format_exc()
    (EVIDENCE / "failure.txt").write_text(failure)
    print(failure, flush=True)

manifest = {"candidate_code_commit": CODE_COMMIT, "reference_commit": REFERENCE_COMMIT,
            "python": platform.python_version(), "stages": STAGES,
            "scope": "synthetic software and numerical regression; no patient accuracy qualification",
            "all_commands_passed": failure is None and all(s["returncode"] == 0 for s in STAGES),
            "failure": failure,
            "source_sha256": {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()
                              for p in sorted((REPO / "src").rglob("*.py"))}}
(EVIDENCE / "manifest.json").write_text(json.dumps(manifest, indent=2))
archive = ROOT / "mixed_mini_assembly_evidence.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in EVIDENCE.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(EVIDENCE))
receipt = {"path": str(archive), "bytes": archive.stat().st_size,
           "sha256": hashlib.sha256(archive.read_bytes()).hexdigest()}
print("RECEIPT", json.dumps(receipt), flush=True)
print("ALL_COMMANDS_PASSED", manifest["all_commands_passed"], flush=True)
# Offer the archive on both success and failure, before raising the final gate.
# A failed setup or regression must not prevent retrieval of its own diagnostics.
from google.colab import files
try:
    files.download(str(archive))
except Exception:
    print("Browser download could not be offered; archive remains at", archive)
    print(traceback.format_exc())
assert manifest["all_commands_passed"], "Failed replay: retrieve the evidence archive and diagnose before merge."


virtualenv-install returncode 0

venv returncode 0
created virtual environment CPython3.13.16.final.0-64-x86_64 in 1066ms
  creator CPython3Posix(dest=/content/mini_assembly_20261023_192114_429258/venv, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.2
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator

clone returncode 0
Cloning into '/content/mini_assembly_20261023_192114_429258/CardiMech'...

checkout returncode 0
Note: switching to '584785c32caadd3c6bc42fef712b3762789add85'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may


In [2]:
import xml.etree.ElementTree as ET
export_script = """from pathlib import Path
import json, numpy as np, hashlib, sys
root=Path(sys.argv[1]); identities={}
keys=['reference_nodes','tetrahedra','displacement','mixed_pressure_Pa','bubble_displacement_m','element_J']
for p in sorted((root/'qualification').rglob('spatial-mechanics.json')):
    raw=p.read_bytes(); data=json.loads(raw)
    np.savez_compressed(p.with_name('state.npz'),**{k:np.asarray(data[k]) for k in keys if k in data})
    mesh=p.with_name('mesh.json')
    identities[str(p.relative_to(root))]={'field_sha256':hashlib.sha256(raw).hexdigest(),'field_bytes':len(raw),'mesh_sha256':hashlib.sha256(mesh.read_bytes()).hexdigest()}
(root/'bulk-field-identities.json').write_text(json.dumps(identities,indent=2))
"""
(ROOT/'export_states.py').write_text(export_script)
stage("state-export",[PY,str(ROOT/'export_states.py'),str(EVIDENCE)],cwd=REPO)
suite_counts={k:0 for k in ('tests','failures','errors','skipped')}
for s in ET.parse(EVIDENCE/'junit.xml').getroot().iter('testsuite'):
    for k in suite_counts: suite_counts[k]+=int(s.attrib.get(k,0))
qualification=json.loads((EVIDENCE/'qualification/qualification.json').read_text())
comparison=json.loads((EVIDENCE/'comparison/comparison.json').read_text())
identities=json.loads((EVIDENCE/'bulk-field-identities.json').read_text())
assert len(identities)==23, len(identities)
compact_manifest={'replay':manifest,'suite_counts':suite_counts,'qualification':qualification,
    'comparison':comparison,'full_archive_receipt':receipt,'bulk_field_identities':identities,
    'raw_state_count':len(identities),
    'omissions':'Full mesh/element field JSON omitted from compact archive; hashes retained and 23 raw nodal/pressure/bubble/J states included for regeneration with pinned source. Full archive was offered separately.'}
(EVIDENCE/'COMPACT_MANIFEST.json').write_text(json.dumps(compact_manifest,indent=2)+'\n')
compact_archive=ROOT/'mixed_mini_assembly_compact_evidence.zip'
with zipfile.ZipFile(compact_archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EVIDENCE.rglob('*')):
        if p.is_file() and p.name not in {'mesh.json','spatial-mechanics.json'}:
            z.write(p,str(p.relative_to(EVIDENCE)))
    for directory in ('src','tests','scripts'):
        for p in sorted((REPO/directory).rglob('*.py')):
            z.write(p,'source/'+str(p.relative_to(REPO)))
    for name in ('MIXED_MINI_ACCEPTANCE.md','MIXED_MINI_DIAGNOSIS.md'):
        z.write(REPO/'docs'/name,'source/docs/'+name)
raw=compact_archive.read_bytes()
compact_receipt={'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest(),
    'git_blob':hashlib.sha1(b'blob '+str(len(raw)).encode()+b'\0'+raw).hexdigest()}
print('SUMMARY',json.dumps({'suite':suite_counts,'qualification_passed':qualification['passed'],
    'acceptance':qualification['acceptance'],'comparisons':comparison['rows'],
    'compact_receipt':compact_receipt,'full_receipt':receipt}),flush=True)


state-export returncode 0
SUMMARY {"suite": {"tests": 152, "failures": 0, "errors": 0, "skipped": 0}, "qualification_passed": true, "acceptance": {"all_equilibria_passed": true, "finest_all_regimes_within_original_15_percent": true, "near_incompressible_error_improved_at_least_fourfold": true, "refinement_errors_decrease": true, "finest_quadrature_change_below_one_percent": true, "low_load_scaled_displacement_change_below_one_percent": true, "higher_bulk_error_within_original_15_percent": true, "finite_load_quadrature_change_below_one_percent": true}, "comparisons": [{"model": "neo_hookean", "level": 1, "nodes": 72, "tetrahedra": 288, "tangent_relative_error": 3.17371994669411e-16, "all_state_checks_passed": true, "seconds": {"reference": [2.152851188999989, 2.1456071300000303, 2.1541417329999604], "candidate": [0.26476770000010674, 0.2655612190000056, 0.2581817529999171]}, "median_speedup": 8.131094499061332}, {"model": "neo_hookean", "level": 2, "nodes": 330, "tetrahedra": 1536, "tan